# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [19]:
%load_ext dotenv
%dotenv 



In [20]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [21]:
import os
from glob import glob

PRICE_DATA = os.getenv("PRICE_DATA") #load PRICE_DATA
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)

dd_px = dd.read_parquet(parquet_files).set_index("ticker")



For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [22]:
dd_features = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), 
                       Adj_Close_lag_1 = x['Adj Close'].shift(1),
                       Returns = x['Close'] / x['Close'].shift(1) - 1,
                       Hi_Lo_Range = x['High'] - x['Low'], 
                       )
)

dd_features.head()



/var/folders/vx/9l926v794597r432fpxr3m700000gn/T/ipykernel_90561/2710476468.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_features = dd_px.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_Range
ticker,,,,,,,,,,,,,
APYX,2019-01-02,7.96,7.960,6.211,6.47,6.47,104900,APYX.csv,2019,NaN,NaN,NaN,1.749
APYX,2019-01-03,6.75,6.750,6.100,6.50,6.50,130400,APYX.csv,2019,6.47,6.47,0.004637,0.650
APYX,2019-01-04,6.50,6.975,6.450,6.62,6.62,108700,APYX.csv,2019,6.50,6.50,0.018462,0.525
APYX,2019-01-07,6.84,8.530,6.750,7.49,7.49,401900,APYX.csv,2019,6.62,6.62,0.131420,1.780
APYX,2019-01-08,7.50,8.100,7.493,7.74,7.74,299400,APYX.csv,2019,7.49,7.49,0.033378,0.607


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [23]:
pd_features = dd_features.compute() 
pd_features['Returns_Rolling_Avg10'] = pd_features.groupby('ticker')['Returns'].transform(lambda x: x.rolling(10, min_periods=1).mean())
pd_features.head()



,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_Range,Returns_Rolling_Avg10
ticker,,,,,,,,,,,,,,
APYX,2019-01-02,7.96,7.960,6.211,6.47,6.47,104900.0,APYX.csv,2019,NaN,NaN,NaN,1.749,NaN
APYX,2019-01-03,6.75,6.750,6.100,6.50,6.50,130400.0,APYX.csv,2019,6.47,6.47,0.004637,0.650,0.004637
APYX,2019-01-04,6.50,6.975,6.450,6.62,6.62,108700.0,APYX.csv,2019,6.50,6.50,0.018462,0.525,0.011549
APYX,2019-01-07,6.84,8.530,6.750,7.49,7.49,401900.0,APYX.csv,2019,6.62,6.62,0.131420,1.780,0.051506
APYX,2019-01-08,7.50,8.100,7.493,7.74,7.74,299400.0,APYX.csv,2019,7.49,7.49,0.033378,0.607,0.046974


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

No, it's not necessary to convert to Pandas just to compute a moving average. 
However because the Dask DataFrame would require grouping by ticker and then applying a rolling function over date within ticker, it might be more efficient to compute in Pandas for a small dataset like this.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.